In [1]:
import pandas as pd 
import numpy as np

In [2]:
dtype_map = {
    'user_id'      : 'str',
    'action'       : 'category',
    'action_type'  : 'category',
    'action_detail': 'category',
    'device_type'  : 'category',
    'secs_elapsed' : 'float32'  
}

sessions = pd.read_csv('data/session_cleaned.csv' ,dtype=dtype_map,usecols=['user_id', 'action', 'action_type', 'device_type', 'secs_elapsed']
    # action_detail dropped -> too noisy
)
print("Sessions loaded.")



Sessions loaded.


In [ ]:
device_map = {
    'Mac Desktop'              : 'desktop',
    'Windows Desktop'          : 'desktop',
    'Chromebook'               : 'desktop',
    'Linux Desktop'            : 'desktop',
    'iPhone'                   : 'mobile',
    'Android Phone'            : 'mobile',
    'Mobile Safari'            : 'mobile',
    'Android App Unknown Phone/Tablet': 'mobile',
    'iPad Tablet'              : 'tablet',
    'Android Tablet'           : 'tablet',
    'Tablet'                   : 'tablet', 
    '-unknown-'                : 'unknown'
}

sessions['device_type'] = (sessions['device_type'].astype(str).map(device_map).fillna('unknown').astype('category'))

In [4]:
agg = sessions.groupby('user_id').agg(
    total_actions        = ('action','count'),
    total_seconds        = ('secs_elapsed','sum'),
    unique_actions       = ('action','nunique'),  
    unique_action_types  = ('action_type', 'nunique'),
    unique_devices       = ('device_type', 'nunique'),
).reset_index()

In [5]:
most_used_device = (sessions.groupby(['user_id', 'device_type']).size().reset_index(name='count').sort_values('count', ascending=False)
    .drop_duplicates(subset='user_id')  
    [['user_id', 'device_type']]
    .rename(columns={'device_type': 'most_used_device'})
)


In [14]:
most_used_device['most_used_device'].value_counts()

most_used_device
desktop    82335
mobile     37022
tablet     10339
unknown     5787
Name: count, dtype: int64

In [15]:
sessions['action'].value_counts()

action
show                         2758985
index                         841071
search_results                723124
personalize                   704782
search                        533833
                              ...   
host_cancel                        1
events                             1
revert_to_admin                    1
set_minimum_payout_amount          1
deactivated                        1
Name: count, Length: 360, dtype: int64

In [16]:
booking_keywords = ['book', 'payment', 'checkout', 'reservation', 'request']

sessions['is_booking_action'] = (sessions['action'].astype(str).str.contains('|'.join(booking_keywords), case=False, na=False).astype('int8'))

booking_intent = (
    sessions
    .groupby('user_id')['is_booking_action']
    .sum()
    .reset_index()
    .rename(columns={'is_booking_action': 'booking_intent_count'})
)

In [17]:
final_sessions = agg.copy()
final_sessions = final_sessions.merge(most_used_device, on='user_id', how='left')
final_sessions = final_sessions.merge(booking_intent,   on='user_id', how='left')

In [18]:
# Derived features
final_sessions['avg_seconds_per_action'] = (final_sessions['total_seconds'] / final_sessions['total_actions']).replace([np.inf, np.nan], 0)

In [20]:

del sessions   # frees ~700MB immediately
print(f"Sessions aggregated: {final_sessions.shape[0]:,} users | {final_sessions.shape[1]} features")

final_sessions.to_csv('data/session_aggregated.csv', index=False)
print("Saved to session_aggregated.csv  ")

Sessions aggregated: 135,483 users | 9 features
Saved to session_aggregated.csv  


In [21]:
df = pd.read_csv('data/session_aggregated.csv')
df.head()

,user_id,total_actions,total_seconds,unique_actions,unique_action_types,unique_devices,most_used_device,booking_intent_count,avg_seconds_per_action
0,00023iyk9l,40,867896.0,14,7,2,desktop,5,21697.400000
1,0010k6l0om,63,586543.0,11,5,1,desktop,0,9310.206349
2,001wyh0pz8,90,282965.0,10,5,1,mobile,0,3144.055556
3,0028jgx1x1,31,297010.0,5,5,2,unknown,0,9580.967742
4,002qnbzfs5,789,6487080.0,26,7,2,mobile,22,8221.901141
